# PPE Detection Model Training: SH17 + Primary Dataset Integration
### 8-Class Unified Safety Compliance Model

This notebook trains a YOLO model detecting all 8 core industrial safety categories:
1. **Hard hat** (`1: helmet`)
2. **Safety glasses** (`5: goggles`)
3. **Safety vest** (`2: safety_vest`)
4. **Gloves** (`3: gloves`)
5. **Safety boots** (`4: safety_shoes`)
6. **Mask** (`6: mask`)
7. **Ear protection** (`7: ear_protection`)
8. **Person / Worker** (`0: person`)

Datasets Combined:
- **Primary Industrial PPE Dataset** (`Datasets/Processed/PPE_Final`): 3,252 curated images
- **SH17 Dataset** (Pexels industrial safety image crawl): 499 images with 17-class transfer annotations
- **Industrial Face Mask Dataset**: 139 high-resolution annotated mask images
- **Total Combined**: **3,876 images** (2,806 train, 744 val, 326 test)

In [ ]:
import os
import glob
from pathlib import Path
from collections import Counter
from ultralytics import YOLO

DATASET_ROOT = Path('../Datasets/Processed/PPE_Combined_SH17').resolve()
DATA_YAML = DATASET_ROOT / 'data.yaml'
print(f"Dataset root: {DATASET_ROOT}")
print(f"YAML file exists: {DATA_YAML.exists()}")

### 1. Unified 8-Class Distribution & Annotation Statistics

In [ ]:
CLASS_NAMES = ['person', 'helmet', 'safety_vest', 'gloves', 'safety_shoes', 'goggles', 'mask', 'ear_protection']
total_instances = Counter()

for split in ['train', 'val', 'test']:
    imgs = list((DATASET_ROOT / 'images' / split).glob('*.*'))
    lbls = list((DATASET_ROOT / 'labels' / split).glob('*.txt'))
    split_counts = Counter()
    for lp in lbls:
        with open(lp, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    cid = int(parts[0])
                    split_counts[cid] += 1
                    total_instances[cid] += 1
    print(f"Split {split:5s}: {len(imgs):4d} images | {sum(split_counts.values()):5d} annotations")

print("\nTotal Instances Per Class Across All Splits:")
for cid in range(8):
    print(f"  {cid}: {CLASS_NAMES[cid]:16s} = {total_instances[cid]:5d}")

### 2. Model Training Configuration & Execution

In [ ]:
# Load model architecture
model = YOLO('yolov8n.pt')

# Train model on combined dataset
results = model.train(
    data=str(DATA_YAML),
    epochs=3,
    batch=16,
    imgsz=480,
    device='cpu',
    project='../runs',
    name='train_combined_sh17',
    exist_ok=True,
    verbose=True
)
print("Training completed!")

### 3. Model Evaluation on Held-Out Validation Split

In [ ]:
best_weights = Path('../runs/train_combined_sh17/weights/best.pt')
if not best_weights.exists():
    best_weights = Path('../runs/train_combined_sh17/weights/last.pt')

eval_model = YOLO(str(best_weights))
metrics = eval_model.val(data=str(DATA_YAML), split='val')
print(f"Validation mAP50: {metrics.box.map50:.4f}")
print(f"Validation mAP50-95: {metrics.box.map:.4f}")

### 4. Checkpoint Export to Production Path

In [ ]:
import shutil
export_path = Path('../Model/PPE_Combined_SH17_Best.pt')
shutil.copy2(best_weights, export_path)
print(f"Model exported to {export_path} ({export_path.stat().st_size / 1024 / 1024:.2f} MB)")